# EIA Total Crude Production Ingestion

## Purpose
End-to-end data ingestion pipeline that fetches crude oil production data from the EIA (Energy Information Administration) API, loads it to the bronze layer, then cleans and transforms it for the silver layer. This notebook handles both annual and monthly production time series.

## Data Flow

**Data Source (Read):**
* EIA International Energy API - Crude oil and condensate production data (productId: 57, activityId: 1)
  * Frequency: Annual and Monthly
  * Coverage: 1980-2024, all countries
  * Unit: TBPD (Thousand Barrels Per Day)

**Data Destinations (Write):**
* **Bronze Layer** (raw API data):
  * `bronze.eia_oilcond_production_annual` - Raw annual production data
  * `bronze.eia_oilcond_production_monthly` - Raw monthly production data
* **Silver Layer** (cleaned data):
  * `silver.eia_oilcond_production_annual` - Cleaned annual production (1980-2024, country-level only)
  * `silver.eia_oilcond_production_monthly` - Cleaned monthly production (country-level only)

## Workflow

### Annual Production Pipeline
1. **Fetch from API** - POST request with pagination (5000 records per batch) for all country IDs
2. **Load to bronze** - Write raw data to `bronze.eia_oilcond_production_annual`
3. **Transform** - Select required columns, filter 1980-2024, country-level only (countryRegionTypeId = 'c')
4. **Clean** - Remove null values and zeros from production values
5. **Load to silver** - Write cleaned data to `silver.eia_oilcond_production_annual`
6. **Verify** - Display sample data and row count

### Monthly Production Pipeline
7. **Fetch from API** - POST request with pagination for monthly data
8. **Load to bronze** - Write raw data to `bronze.eia_oilcond_production_monthly`
9. **Transform** - Select required columns, filter country-level only
10. **Clean** - Remove null values and zeros
11. **Load to silver** - Write cleaned data to `silver.eia_oilcond_production_monthly`

## Key Features
* **Paginated API calls** - Handles large datasets with 5000-record batches
* **Comprehensive coverage** - 200+ country/region IDs
* **POST requests** - Avoids URI length limits with JSON payload
* **Data quality** - Filters out nulls, zeros, and non-country aggregates
* **Full refresh** - Overwrites tables on each run for consistency
* **Dual frequency** - Separate pipelines for annual and monthly data

In [0]:
import pandas as pd
import requests
from io import StringIO

base_url = "https://api.eia.gov/v2/international/data/"
api_key = "BEvLzF1LIQm3Klx8lfHIJKdAYHkxniZrifcdt0Kx"

# List of all country/region IDs to include
country_ids = [
    "ABW", "AFG", "AFRC", "AGO", "ALB", "ARE", "ARG", "ARM", "ASM", "ATA", "ATG", "AUS", "AUT", "AZE",
    "BDI", "BEL", "BEN", "BFA", "BGD", "BGR", "BHR", "BHS", "BIH", "BLR", "BLZ", "BMU", "BOL", "BRA", "BRB",
    "BRN", "BTN", "BWA", "CAF", "CAN", "CHE", "CHL", "CHN", "CIV", "CMR", "COD", "COG", "COK", "COL", "COM",
    "CPV", "CRI", "CSAM", "CSK", "CUB", "CYM", "CYP", "CZE", "DDR", "DEU", "DEUW", "DJI", "DMA", "DNK", "DOM",
    "DZA", "ECU", "EGY", "ERI", "ESH", "ESP", "EST", "ETH", "FIN", "FJI", "FLK", "FRA",
    "FRO", "GAB", "GBR", "GEO", "GHA", "GIB", "GIN", "GLP", "GMB", "GNB", "GNQ", "GRC", "GRD", "GRL", "GTM",
    "GUF", "GUM", "GUY", "HITZ", "HKG", "HND", "HRV", "HTI", "HUN", "IDN", "IND", "IRL", "IRN", "IRQ", "ISL",
    "ISR", "ITA", "JAM", "JOR", "JPN", "KAZ", "KEN", "KGZ", "KHM", "KIR", "KNA", "KOR", "KWT", "LAO", "LBN",
    "LBR", "LBY", "LCA", "LKA", "LSO", "LTU", "LUX", "LVA", "MAC", "MAR", "MDA", "MDG", "MDV", "MEX",
    "MKD", "MLI", "MLT", "MMR", "MNE", "MNG", "MOZ", "MRT", "MSR", "MTQ", "MUS", "MWI", "MYS", "NAM", "NCL",
    "NER", "NGA", "NIC", "NIU", "NLD", "NLDA", "NOR", "NPL", "NRU", "NZL", "OMN", "PAK", "PAN", "PER", "PERG", "PHL", "PNG", "POL",
    "PRI", "PRK", "PRT", "PRY", "PSE", "PYF", "QAT", "REU", "ROU", "RUS", "RWA", "SAU", "SCG", "SDN", "SEN",
    "SGP", "SHN", "SLB", "SLE", "SLV", "SOM", "SPM", "SRB", "SSD", "STP", "SUN", "SUR", "SVK", "SVN", "SWE",
    "SWZ", "SYC", "SYR", "TCA", "TCD", "TGO", "THA", "TJK", "TKM", "TLS", "TON", "TTO", "TUN", "TUR", "TWN",
    "TZA", "UGA", "UKR", "URY", "USA", "USIQ", "UZB", "VCT", "VEN", "VGB", "VIR", "VNM", "VUT", "WAK",
    "WP13", "WP14", "WP15", "WP16", "WP17", "WP18", "WP24", "WP25", "WP26", "WP27", "WSM", "XKS", "YEM", "YUG", "ZAF", "ZMB", "ZWE"
]

all_dfs = []
offset = 0
length = 5000

while True:
    # Use POST with JSON body to avoid URI length limits
    payload = {
        "frequency": "monthly",
        "data": ["value"],
        "facets": {
            "activityId": ["1"],
            "productId": ["57"],
            "countryRegionId": country_ids,
            "unit": ["TBPD"]
        },
        "sort": [{"column": "period", "direction": "asc"}],
        "offset": offset,
        "length": length
    }
    
    response = requests.post(f"{base_url}?api_key={api_key}", json=payload)
    response.raise_for_status()
    
    data = response.json()
    
    if "response" not in data or "data" not in data["response"]:
        break
    
    records = data["response"]["data"]
    
    if len(records) == 0:
        break
    
    batch_df = pd.DataFrame(records)
    all_dfs.append(batch_df)
    
    if len(records) < length:
        break
    
    offset += length

df = spark.createDataFrame(pd.concat(all_dfs, ignore_index=True))

#df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("bronze.eia_oilcond_production_annual")


---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File <command-7004088867480262>, line 49
     34 payload = {
     35     "frequency": "monthly",
     36     "data": ["value"],
   (...)
     45     "length": length
     46 }
     48 response = requests.post(f"{base_url}?api_key={api_key}", json=payload)
---> 49 response.raise_for_status()
     51 data = response.json()
     53 if "response" not in data or "data" not in data["response"]:

File /databricks/python/lib/python3.12/site-packages/requests/models.py:1024, in Response.raise_for_status(self)
   1019     http_error_msg = (
   1020         f"{self.status_code} Server Error: {reason} for url: {self.url}"
   1021     )
   1023 if http_error_msg:
-> 1024     raise HTTPError(http_error_msg, response=self)

HTTPError: 502 Server Error: Bad Gateway for url: https://api.eia.gov/v2/international/data/?api_key=BEvLzF1LIQm3Klx8lfHIJKdAYHkx

In [0]:
#view united states data in bronze.eia_oilcond_production
display(spark.sql("SELECT * FROM bronze.eia_oilcond_production WHERE countryRegionName = 'United States'"))

period,productId,productName,activityId,activityName,countryRegionId,countryRegionName,countryRegionTypeId,countryRegionTypeName,dataFlagId,dataFlagDescription,unitName,value,unit
1997,57,Crude oil including lease condensate,1,Production,WP27,United States,r,Region,null,null,thousand barrels per day,6451.59223561643835616438356164383561644,TBPD
1998,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,null,null,thousand barrels per day,6251.83396986301369863013698630136986301,TBPD
1998,57,Crude oil including lease condensate,1,Production,WP27,United States,r,Region,null,null,thousand barrels per day,6251.83396986301369863013698630136986301,TBPD
1999,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,null,null,thousand barrels per day,5881.45658356164383561643835616438356164,TBPD
1999,57,Crude oil including lease condensate,1,Production,WP27,United States,r,Region,null,null,thousand barrels per day,5881.45658356164383561643835616438356164,TBPD
2000,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,null,null,thousand barrels per day,5821.60109480874316939890710382513661202,TBPD
2000,57,Crude oil including lease condensate,1,Production,WP27,United States,r,Region,null,null,thousand barrels per day,5821.60109480874316939890710382513661202,TBPD
2001,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,null,null,thousand barrels per day,5801.40273758904109589041095890410958904,TBPD
2001,57,Crude oil including lease condensate,1,Production,WP27,United States,r,Region,null,null,thousand barrels per day,5801.40273758904109589041095890410958904,TBPD
2002,57,Crude oil including lease condensate,1,Production,USA,United States,c,Country,null,null,thousand barrels per day,5744.07671232876534246575342465753424658,TBPD


In [0]:
from delta.tables import DeltaTable

# Load the Delta table metadata
pandas_df = spark.sql("""
    SELECT 
        countryRegionName as Country, 
        period as Year, 
        productName, 
        unitName, 
        TRY_CAST(value AS FLOAT) as value
    FROM bronze.eia_oilcond_production
    WHERE period BETWEEN 1980 AND 2024 AND countryRegionTypeId = "c"
""").toPandas()

#drop all rows with null values or zeros from the 'value' column
pandas_df = pandas_df.dropna(subset=['value'])
pandas_df = pandas_df[pandas_df['value'] != 0]

#convert pandas_df to spark_df
spark_df = spark.createDataFrame(pandas_df)

spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "silver.eia_oilcond_production_annual"
)

In [0]:
# Verify the data
df_sample = spark.table("silver.eia_oilcond_production")
print(f"Total rows: {df_sample.count()}")
display(df_sample.limit(100))

Total rows: 3729


Country,Year,productName,unitName,value
Greece,2016,Crude oil including lease condensate,thousand barrels per day,3.172131
Guatemala,2016,Crude oil including lease condensate,thousand barrels per day,9.776486
Croatia,2016,Crude oil including lease condensate,thousand barrels per day,13.581967
Hungary,2016,Crude oil including lease condensate,thousand barrels per day,13.833333
Indonesia,2016,Crude oil including lease condensate,thousand barrels per day,831.62024
India,2016,Crude oil including lease condensate,thousand barrels per day,737.2279
Iran,2016,Crude oil including lease condensate,thousand barrels per day,4151.1665
Iraq,2016,Crude oil including lease condensate,thousand barrels per day,4451.5166
Israel,2016,Crude oil including lease condensate,thousand barrels per day,0.39
Italy,2016,Crude oil including lease condensate,thousand barrels per day,70.674866


#Do the same for the monthly dataset

In [0]:
base_url = "https://api.eia.gov/v2/international/data/"
api_key = "BEvLzF1LIQm3Klx8lfHIJKdAYHkxniZrifcdt0Kx"

# List of all country/region IDs to include
country_ids = [
    "ABW", "AFG", "AFRC", "AGO", "ALB", "ARE", "ARG", "ARM", "ASM", "ATA", "ATG", "AUS", "AUT", "AZE",
    "BDI", "BEL", "BEN", "BFA", "BGD", "BGR", "BHR", "BHS", "BIH", "BLR", "BLZ", "BMU", "BOL", "BRA", "BRB",
    "BRN", "BTN", "BWA", "CAF", "CAN", "CHE", "CHL", "CHN", "CIV", "CMR", "COD", "COG", "COK", "COL", "COM",
    "CPV", "CRI", "CSAM", "CSK", "CUB", "CYM", "CYP", "CZE", "DDR", "DEU", "DEUW", "DJI", "DMA", "DNK", "DOM",
    "DZA", "ECU", "EGY", "ERI", "ESH", "ESP", "EST", "ETH", "FIN", "FJI", "FLK", "FRA",
    "FRO", "GAB", "GBR", "GEO", "GHA", "GIB", "GIN", "GLP", "GMB", "GNB", "GNQ", "GRC", "GRD", "GRL", "GTM",
    "GUF", "GUM", "GUY", "HITZ", "HKG", "HND", "HRV", "HTI", "HUN", "IDN", "IND", "IRL", "IRN", "IRQ", "ISL",
    "ISR", "ITA", "JAM", "JOR", "JPN", "KAZ", "KEN", "KGZ", "KHM", "KIR", "KNA", "KOR", "KWT", "LAO", "LBN",
    "LBR", "LBY", "LCA", "LKA", "LSO", "LTU", "LUX", "LVA", "MAC", "MAR", "MDA", "MDG", "MDV", "MEX",
    "MKD", "MLI", "MLT", "MMR", "MNE", "MNG", "MOZ", "MRT", "MSR", "MTQ", "MUS", "MWI", "MYS", "NAM", "NCL",
    "NER", "NGA", "NIC", "NIU", "NLD", "NLDA", "NOR", "NPL", "NRU", "NZL", "OMN", "PAK", "PAN", "PER", "PERG", "PHL", "PNG", "POL",
    "PRI", "PRK", "PRT", "PRY", "PSE", "PYF", "QAT", "REU", "ROU", "RUS", "RWA", "SAU", "SCG", "SDN", "SEN",
    "SGP", "SHN", "SLB", "SLE", "SLV", "SOM", "SPM", "SRB", "SSD", "STP", "SUN", "SUR", "SVK", "SVN", "SWE",
    "SWZ", "SYC", "SYR", "TCA", "TCD", "TGO", "THA", "TJK", "TKM", "TLS", "TON", "TTO", "TUN", "TUR", "TWN",
    "TZA", "UGA", "UKR", "URY", "USA", "USIQ", "UZB", "VCT", "VEN", "VGB", "VIR", "VNM", "VUT", "WAK",
    "WP13", "WP14", "WP15", "WP16", "WP17", "WP18", "WP24", "WP25", "WP26", "WP27", "WSM", "XKS", "YEM", "YUG", "ZAF", "ZMB", "ZWE"
]

all_dfs = []
offset = 0
length = 5000

while True:
    # Use POST with JSON body to avoid URI length limits
    payload = {
        "frequency": "monthly",
        "data": ["value"],
        "facets": {
            "activityId": ["1"],
            "productId": ["57"],
            "countryRegionId": country_ids,
            "unit": ["TBPD"]
        },
        "sort": [{"column": "period", "direction": "asc"}],
        "offset": offset,
        "length": length
    }
    
    response = requests.post(f"{base_url}?api_key={api_key}", json=payload)
    response.raise_for_status()
    
    data = response.json()
    
    if "response" not in data or "data" not in data["response"]:
        break
    
    records = data["response"]["data"]
    
    if len(records) == 0:
        break
    
    batch_df = pd.DataFrame(records)
    all_dfs.append(batch_df)
    
    if len(records) < length:
        break
    
    offset += length

df_bronze = spark.createDataFrame(pd.concat(all_dfs, ignore_index=True))


df_bronze.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("bronze.eia_oilcond_production_monthly")

In [0]:
# Load the Delta table metadata
pandas_df = spark.sql("""
    SELECT 
        countryRegionName as Country, 
        period as Date, 
        productName, 
        unitName, 
        TRY_CAST(value AS FLOAT) as value
    FROM bronze.eia_oilcond_production_monthly
    WHERE countryRegionTypeId = "c"
""").toPandas()

#drop all rows with null values or zeros from the 'value' column
pandas_df = pandas_df.dropna(subset=['value'])
pandas_df = pandas_df[pandas_df['value'] != 0]

#convert pandas_df to spark_df
spark_df = spark.createDataFrame(pandas_df)

spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "silver.eia_oilcond_production_monthly"
)